# House Stage 1 regression

The formula is the model: `run_ols(formula, train_years, eval_years)` fits it with `smf.ols(formula, data=train).fit()` and returns the native statsmodels result as `result.model`. It is the same function as `PYTHONPATH=src python -m house.experiments.run` (see `src/house/experiments/run.py`). Shared data prep, validation, and metrics are in `src/house/experiments/evaluation.py`, and saved runs in `src/house/experiments/tracking.py`. Holdout years: `src/house/regression/config.py`. Columns: `src/house/regression/schema.py`.
Build the predictors and targets first: `python data-processing/house/house_features.py` (`modeling.csv`) and `python data-processing/house/house_targets.py` (`targets.csv`).
Imputed targets are never trained on or scored: they were built from partisan lean, so scoring against them would reward a model for agreeing with it. Evaluation scores valid targets only (observed, non-imputed, of an eligible race type) and competitive subsets of them, per eval year; see `src/house/experiments/evaluation.py`. Every left-out row is listed with its reason in `result.excluded`. Nothing is saved unless you ask (`save=True` or `tracking.save_run`).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from house.experiments import evaluation, tracking
from house.experiments.run import run_ols

In [ ]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [ ]:
print(hr.validate_schema(targets, schema=hr.TARGETS_SCHEMA).summary())

In [ ]:
FORMULA = "adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed"
TRAIN_YEARS = [2006, 2008, 2010, 2012, 2014, 2016]
EVAL_YEARS = [2018, 2020, 2022]     # scored against actual targets
PREDICT_YEARS = []      # predicted only, no metrics, e.g. [2026] for a forecast
USE_HOLDOUT = False     # True allows HOLDOUT_YEARS (2024) in training, evaluation, or prediction
REQUIRE_COMPLETE = []   # extra columns a row must have, so runs with different formulas share rows
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}

result = run_ols(
    formula=FORMULA,
    train_years=TRAIN_YEARS,
    eval_years=EVAL_YEARS,
    predict_years=PREDICT_YEARS,
    predictors=predictors,  # the tables loaded above, instead of reloading per run
    targets=targets,
    use_holdout=USE_HOLDOUT,
    require_complete=REQUIRE_COMPLETE,
    race_types=EVAL_RACE_TYPES,
    competitive=COMPETITIVE,
    name="classic_ols",
    save=False,             # True writes results/house/runs/<run_id>
)
print(tracking.describe(result))  # result.config has the raw values

In [ ]:
evaluation.excluded_by_year(result.excluded, max_districts=5)  # every row left out, by year and reason (None lists all)

In [ ]:
result.model.summary()

In [ ]:
result.metrics.round(3)  # by (subset, year), then pooled ("all"); bias = mean(actual - predicted): + means Democrats outperformed

In [ ]:
# Winner calls, in % of the classified races (ties left out): "valid" is every observed, non-imputed district
#   accuracy_pct     correct winner calls
#   precision_D_pct  of the races predicted D, the % the Democrat won
#   precision_R_pct  of the races predicted R, the % the Republican won
result.metrics[["n_classified", "accuracy_pct", "precision_D_pct", "precision_R_pct"]].round(1)

## Stage 1 evaluation

`result.metrics` scores the **valid** targets (a finite `adj_margin_2p_norm`, `is_imputed == False`, race type in `EVAL_RACE_TYPES`) and each `COMPETITIVE` subset of them, per eval year and pooled (`all`). An empty subset shows `n = 0` and NaN metrics.

- `residual = actual - predicted` everywhere (`result.predictions`, `result.scored`, the metrics, the reports below, and statsmodels' `result.model.resid`). `bias` is the mean residual: **positive means Democrats outperformed the forecast**, negative that Republicans did.
- Winners come from the sign of each margin. A margin of exactly 0 is a tie, not assigned to a party: such rows are left out of `n_classified`, the denominator of `missed_pct` (the percentage of classified races with the wrong predicted winner), and counted in `n_actual_tie` / `n_predicted_tie`.
- `n_winner_conflict` counts rows whose actual margin sign disagrees with `targets.winner_party`, e.g. AK-AL 2022, an RCV race the Democrat won after trailing in the first round. Classification still uses the margin sign; the rows are flagged in `result.scored["winner_conflict"]`.

`result.coverage` accounts for every eval-year district: excluded by reason (first match wins: `missing_target`, `imputed_target`, `race_type:<type>`, `no_prediction`), then `n_valid` and the size of each competitive subset.

In [ ]:
result.coverage  # eval districts by exclusion reason, then the scored subsets

In [ ]:
# actual (rows) by predicted (columns) winner, pooled over the eval years; ties are left out (counted in metrics).
# As counts, then as a percentage of each subset's classified races (the off-diagonal cells add up to missed_pct)
SUBSETS = ["valid", *COMPETITIVE]
display(pd.concat({subset: evaluation.confusion_matrix(result.scored, subset=subset) for subset in SUBSETS}, axis=1))
display(pd.concat({subset: evaluation.confusion_matrix(result.scored, subset=subset, percent=True)
                   for subset in SUBSETS}, axis=1).round(1))

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Sequential blue ramp (light = few races, dark = many), on one shared % scale so the panels compare directly
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
SURFACE, INK, INK_SECONDARY = "#fcfcfb", "#0b0b0b", "#52514e"
MISS_LABELS = {("D", "R"): "Dem miss", ("R", "D"): "GOP miss"}  # (actual, predicted)
CONFUSION_YEAR = None  # one eval year, e.g. 2022; None pools every eval year


def plot_confusion(scored, subsets, year=None):
    """One heatmap per subset: actual (rows) by predicted (columns) winner, each cell's count and its % of the
    subset's classified races (ties left out, as in metrics)."""
    counts = {s: evaluation.confusion_matrix(scored, subset=s, year=year) for s in subsets}
    shares = {s: evaluation.confusion_matrix(scored, subset=s, year=year, percent=True) for s in subsets}
    vmax = pd.concat(shares.values()).max().max()
    vmax = vmax if vmax > 0 else 100  # every subset empty: any scale
    fig, axes = plt.subplots(1, len(subsets), figsize=(3.4 * len(subsets) + 1, 4.2), layout="constrained",
                             facecolor=SURFACE, squeeze=False)
    for ax, subset in zip(axes[0], subsets):
        count, share = counts[subset], shares[subset]
        mesh = ax.pcolormesh(share.to_numpy(), cmap=BLUES, vmin=0, vmax=vmax, edgecolors=SURFACE, linewidth=2)
        for i, actual in enumerate(count.index):
            for j, predicted in enumerate(count.columns):
                pct = share.iloc[i, j]
                r, g, b, _ = BLUES(0 if pd.isna(pct) else pct / vmax)
                ink = SURFACE if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.45 else INK  # readable on the fill
                ax.text(j + 0.5, i + 0.47, f"{count.iloc[i, j]}", ha="center", va="center", fontsize=15,
                        fontweight="bold", color=ink)
                ax.text(j + 0.5, i + 0.70, "-" if pd.isna(pct) else f"{pct:.1f}%", ha="center", va="center",
                        fontsize=10, color=ink)
                if (actual, predicted) in MISS_LABELS:
                    ax.text(j + 0.5, i + 0.22, MISS_LABELS[actual, predicted], ha="center", va="center",
                            fontsize=8.5, color=ink)
        n = int(count.to_numpy().sum())
        missed = int(count.loc["D", "R"] + count.loc["R", "D"])
        ax.set_title(f"{subset}\n{n} races, {missed} missed ({100 * missed / n:.1f}%)" if n
                     else f"{subset}\nno classified races", loc="left", fontsize=10.5, color=INK)
        ax.set_xticks([0.5, 1.5], ["D", "R"])
        ax.set_yticks([0.5, 1.5], ["D", "R"])
        ax.invert_yaxis()  # actual D on top
        ax.set_xlabel("predicted winner", color=INK_SECONDARY)
        ax.set_ylabel("actual winner", color=INK_SECONDARY)
        ax.tick_params(length=0, colors=INK_SECONDARY, labelsize=11)
        ax.spines[:].set_visible(False)
    colorbar = fig.colorbar(mesh, ax=axes[0], shrink=0.75, pad=0.02)
    colorbar.set_label("% of the subset's classified races", color=INK_SECONDARY)
    colorbar.ax.tick_params(length=0, colors=INK_SECONDARY)
    colorbar.outline.set_visible(False)
    fig.suptitle(f"{result.config['name']}: winner calls, " + (str(year) if year else "every eval year"),
                 fontsize=12, color=INK)
    plt.show()


plot_confusion(result.scored, subsets=["valid", *COMPETITIVE], year=CONFUSION_YEAR)

In [ ]:
N_EXAMPLES = 10
EXAMPLE_SUBSET = "competitive_ex_ante"
EXAMPLE_SEED = 0          # a random sample, reproducible: not chosen by how well the model did
EXAMPLE_DISTRICTS = None  # or explicit (year, state, district) keys in EXAMPLE_SUBSET, e.g. [(2022, "AZ", 6)]

evaluation.competitive_examples(result.scored, subset=EXAMPLE_SUBSET, n=N_EXAMPLES, seed=EXAMPLE_SEED,
                                districts=EXAMPLE_DISTRICTS)

In [ ]:
N_WORST = 10

evaluation.worst_misses(result.scored, n=N_WORST)  # every valid target, largest abs_residual first

In [ ]:
for subset in COMPETITIVE:  # the worst misses within each competitive subset
    print(subset)
    display(evaluation.worst_misses(result.scored, n=N_WORST, subset=subset))

### Competitive races by outcome

For each competitive subset: the races predicted correctly, the **Dem misses** (predicted R, the Democrat won), and the **GOP misses** (predicted D, the Republican won). States are in alphabetical order, and within each state the closest actual margin (`abs(adj_margin_2p_norm)`) comes first. The party in parentheses is the actual winner.

In [ ]:
for subset in COMPETITIVE:
    print(evaluation.outcome_report(result.scored, subset=subset), end="\n\n")

In [ ]:
N_STATES = 15

# every valid district by state, pooled over the eval years, worst |mean residual| first (residual = actual - predicted:
# + means Democrats outperformed the forecast). A state with few districts can rank high on one race: read
# mean_residual with n_districts
evaluation.state_summary(result.scored, subsets=COMPETITIVE).head(N_STATES).round(2)

## Prediction diagnostics

The evaluation rows of `result.predictions` (`role == "eval"`, here the 2022 districts), with an observed target. Modeled imputations (uncontested and similar races, see `MODELED_IMPUTATION_METHODS`) are excluded and listed in `result.excluded`; jungle and first-round RCV races are observed, so they are kept. Each row has:

- `prediction`: the model's predicted margin, D positive, in points.
- `actual`: the target, `adj_margin_2p_norm`.
- `residual` = `actual - prediction`, the statsmodels convention (`result.model.resid`) and the sign of the metrics' `bias`: a positive residual means the Democrat outperformed the forecast.

**Reading the two cells below:**

- The residual summary describes the scored rows.
- The scatter plots actual against predicted. Points on the grey diagonal are exact; points above it have a positive residual (the Democrat outperformed the forecast).

**Out-of-range predictions:** OLS is unbounded, so a prediction can fall outside the ±100 a margin can take, typically in the safest seats. Predictions are never clipped here, so the backtest scores the raw model. An evaluation or prediction row beyond ±100 raises a warning (in-sample fitted values aren't flagged).

**Clipping at forecast time:** clipping to ±100 belongs to the forecast step, as one visible line. Every actual margin lies within ±100, so clipping can only move a prediction closer to it. The unclipped MAE and RMSE above are therefore a slight upper bound on the clipped forecast's error.

In [ ]:
scored = result.predictions[result.predictions["role"] == "eval"].dropna(subset=["actual"])
scored["residual"].describe()  # residual = actual - prediction

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(scored["prediction"], scored["actual"], s=14, color="#2a78d6", alpha=0.7, linewidths=0)
lim = [scored[["prediction", "actual"]].min().min(), scored[["prediction", "actual"]].max().max()]
ax.plot(lim, lim, color="#888", lw=1, zorder=0)
ax.set(xlabel="predicted margin (D +)", ylabel="actual margin (D +)", title=f"{result.config['name']}: actual vs. predicted")
ax.grid(alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## Ablations and variants (not saved)

Each block is a dict of formulas whose **first entry is the base** every other run is compared with (`change`). `require_complete` is every predictor column any formula in the dict uses (`evaluation.shared_columns`), so all runs fit and score the same rows (`same rows` ✓): a metric change is a modeling change, not a sample change.

The table shows each subset in `COMPARE_SUBSETS`: every valid (observed, non-imputed) district, then the competitive subsets as defined by `COMPETITIVE` in the settings cell above. For each: `n`, MAE, RMSE, `bias` (mean residual, actual − predicted: **+ means Democrats outperformed the forecast**), and `accuracy_pct` (% correct winner calls). The best value per subset is in bold.

In [ ]:
TARGET = "adj_margin_2p_norm"
COMPARE_SUBSETS = ["valid", *COMPETITIVE]  # the subsets each comparison table shows

ABLATIONS = {
    "classic_ols": FORMULA,  # the base
    "no_gen_margin_raw": f"{TARGET} ~ partisan_lean + inc_signed",
    "no_partisan_lean": f"{TARGET} ~ gen_margin_raw + inc_signed",
    "no_inc_signed": f"{TARGET} ~ gen_margin_raw + partisan_lean",
    "plus_acs": FORMULA + " + college + hispanic",
    "lean_x_inc": FORMULA + " + partisan_lean:inc_signed",
    "inc_categorical": f"{TARGET} ~ gen_margin_raw + partisan_lean + C(inc_signed)",
}
ABLATION_SHARED_COLUMNS = evaluation.shared_columns(formulas=ABLATIONS.values(), columns=predictors.columns)

ablations = [
    run_ols(
        formula=formula,
        train_years=TRAIN_YEARS,
        eval_years=EVAL_YEARS,
        predict_years=PREDICT_YEARS,
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=ABLATION_SHARED_COLUMNS,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=name,
        save=False,
    )
    for name, formula in ABLATIONS.items()
]
tracking.highlight(tracking.compare(ablations, subsets=COMPARE_SUBSETS))

In [ ]:
# each run's coefficients (rows: terms, columns: runs); blank where a run doesn't use the term
pd.concat({run.config["name"]: run.coefficients["coef"] for run in ablations}, axis=1).round(3)

In [ ]:
# District by district, the base run's 10 worst misses: the actual target, then each run's residual (actual -
# prediction; + means Democrats outperformed that run's forecast). Sort by another run's column, or filter to one
# district, e.g. residuals.query("state == 'CA' and district == 13")
residuals = tracking.compare_predictions(ablations, values="residual")
residuals.sort_values("classic_ols", key=abs, ascending=False).head(10)

## FEC fundraising (`fec_f`)

`fec_f.csv` comes from `data/processed/house_fec.csv` (`data-processing/fec_finance.py`), prepared by `house_features.py`:

- `fundraising_advantage`, `spending_advantage`: `log1p(D) - log1p(R)` of receipts / disbursements, capped at ±5 (a 148:1 ratio). Uncapped, a funded nominee against one with no money gives about ±14, which would dominate the fit.
- `small_dollar_share_diff`: the D − R unitemized share of individual contributions. Blank in 2006 (not in that year's FEC file), and wherever a share is undefined.

**Caveats:**

- **Timing:** 2006–2024 totals are FEC's full-cycle summaries, through December 31, so they include post-election money. 2026 runs through mid-October. The model learns from end-of-cycle money and forecasts from pre-election money.
- **Excluded rows:** a few rows are blank and left out (`missing:<var>` in `excluded`): six 2022–2024 winners FEC has no data for (`FEC_KNOWN_GAPS`) and the 2026 races with an unresolved nominee.

The FEC variants share their rows (`FEC_SHARED_COLUMNS`) and show the same subsets, as in the ablations above.

In [ ]:
FEC_ABLATIONS = {
    "classic_ols": FORMULA,  # the base
    "plus_fundraising": FORMULA + " + fundraising_advantage",
    "plus_spending": FORMULA + " + spending_advantage",
    "plus_fundraising_spending": FORMULA + " + fundraising_advantage + spending_advantage",
    "plus_fundraising_no_inc": f"{TARGET} ~ gen_margin_raw + partisan_lean + fundraising_advantage",
}
FEC_SHARED_COLUMNS = evaluation.shared_columns(formulas=FEC_ABLATIONS.values(), columns=predictors.columns)

fec_ablations = [
    run_ols(
        formula=formula,
        train_years=TRAIN_YEARS,
        eval_years=EVAL_YEARS,
        predict_years=PREDICT_YEARS,
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=FEC_SHARED_COLUMNS,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=name,
        save=False,
    )
    for name, formula in FEC_ABLATIONS.items()
]
tracking.highlight(tracking.compare(fec_ablations, subsets=COMPARE_SUBSETS))

In [ ]:
pd.concat({run.config["name"]: run.coefficients["coef"] for run in fec_ablations}, axis=1).round(3)

In [ ]:
# small_dollar_share_diff has no 2006 data, so this comparison trains on TRAIN_YEARS from 2008, on the rows every
# formula can use. plus_fundraising_from_2008 shares those rows, so plus_fundraising_small_dollar against it isolates
# what small_dollar_share_diff adds on top of fundraising_advantage
SMALL_DOLLAR_TRAIN_YEARS = [year for year in TRAIN_YEARS if year >= 2008]
SMALL_DOLLAR_ABLATIONS = {
    "classic_ols_from_2008": FORMULA,  # the base
    "plus_fundraising_from_2008": FORMULA + " + fundraising_advantage",
    "plus_fundraising_small_dollar": FORMULA + " + fundraising_advantage + small_dollar_share_diff",
}
SMALL_DOLLAR_SHARED_COLUMNS = evaluation.shared_columns(formulas=SMALL_DOLLAR_ABLATIONS.values(),
                                                        columns=predictors.columns)

small_dollar_ablations = [
    run_ols(
        formula=formula,
        train_years=SMALL_DOLLAR_TRAIN_YEARS,
        eval_years=EVAL_YEARS,
        predict_years=PREDICT_YEARS,
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=SMALL_DOLLAR_SHARED_COLUMNS,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=name,
        save=False,
    )
    for name, formula in SMALL_DOLLAR_ABLATIONS.items()
]
display(tracking.highlight(tracking.compare(small_dollar_ablations, subsets=COMPARE_SUBSETS)))
# the incremental effect of small_dollar_share_diff: against the fundraising-only variant as the base, same rows
tracking.highlight(tracking.compare(small_dollar_ablations[1:], subsets=COMPARE_SUBSETS))

## Economic conditions (`prices_f`)

`prices_f.csv` holds each district's regional CPI-U and gasoline prices (`data-processing/cpi_gas.py`) and its state's unemployment rate (`data-processing/unemployment.py`), from September of the election year, the last month public before Election Day. See `data-processing/README.md`.

**Signed by the president's party.** Economic conditions are judged against the party holding the White House, so each variable enters as `pres_party_signed:<variable>`. `pres_party_signed` is +1 when a Democrat is president and −1 for a Republican (`PRES_PARTY_SIGNED` below). On the D-margin target, the term moves the Democratic margin by β·x under a Democratic president and the Republican margin by β·x under a Republican one. So **β is the effect on the president's party's margin**: a negative β on inflation means higher inflation hurt the president's party.

Each variable is its own ablation against the base. The levels also vary from year to year, which, with few training years, partly overlaps the generic ballot.

**Relative indicators** keep only the within-year variation, the region or state against the nation:

- **Log ratio** for a positive level: `log_rel_gas_price` = log(regional / national gasoline price). It is compared with the existing dollar gap, `gas_region_minus_us`.
- **Percentage-point difference** for a rate: `unemp_state_minus_us` (state − national unemployment) and `cpi_region_minus_us_yoy` (regional − national inflation), both already in the block.
- **Relative change** for a trend: the 12-month change minus the national 12-month change. That is `gas_region_minus_us_yoy` (compared with `gas_region_yoy`) and `unemp_change_12m_minus_us` (compared with `unemp_change_12m`). For CPI it is again `cpi_region_minus_us_yoy`, since inflation is itself a 12-month change.

Every relative indicator keeps the president's-party sign. All runs share the block's rows, so each pairwise table below compares the relative indicator with the specification it refines on exactly the same districts.

**Caveat:** 2022 regional inflation (about 8.2%) is far above every year in `TRAIN_YEARS` (about 1.2–2.2% over 2010–2020), so the `cpi_region_yoy` variant extrapolates in the eval year.

In [ ]:
# The party holding the White House at each election: +1 Democratic, -1 Republican
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED))

ECON_ABLATIONS = {
    "classic_ols": FORMULA,  # the base
    "pres_x_cpi_region_yoy": FORMULA + " + pres_party_signed:cpi_region_yoy",
    "pres_x_cpi_region_minus_us_yoy": FORMULA + " + pres_party_signed:cpi_region_minus_us_yoy",
    "pres_x_gas_region_yoy": FORMULA + " + pres_party_signed:gas_region_yoy",
    "pres_x_gas_region_minus_us": FORMULA + " + pres_party_signed:gas_region_minus_us",
    "pres_x_unemp_state_rate": FORMULA + " + pres_party_signed:unemp_state_rate",
    "pres_x_unemp_state_minus_us": FORMULA + " + pres_party_signed:unemp_state_minus_us",
    "pres_x_unemp_change_12m": FORMULA + " + pres_party_signed:unemp_change_12m",
    # relative indicators: the region or state against the nation
    "pres_x_log_rel_gas_price": FORMULA + " + pres_party_signed:log_rel_gas_price",  # log ratio of levels
    "pres_x_gas_region_minus_us_yoy": FORMULA + " + pres_party_signed:gas_region_minus_us_yoy",  # relative trend
    "pres_x_unemp_change_12m_minus_us": FORMULA + " + pres_party_signed:unemp_change_12m_minus_us",  # relative trend
}
ECON_SHARED_COLUMNS = evaluation.shared_columns(formulas=ECON_ABLATIONS.values(), columns=predictors.columns)

econ_ablations = [
    run_ols(
        formula=formula,
        train_years=TRAIN_YEARS,
        eval_years=EVAL_YEARS,
        predict_years=PREDICT_YEARS,
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=ECON_SHARED_COLUMNS,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=name,
        save=False,
    )
    for name, formula in ECON_ABLATIONS.items()
]
tracking.highlight(tracking.compare(econ_ablations, subsets=COMPARE_SUBSETS))

In [ ]:
# Each variant's pres_party_signed:<variable> term, one row per variant: the effect on the president's party's margin
# per unit of the variable (e.g. per percentage point of inflation). The OLS standard errors treat districts as
# independent, but these variables vary only by region or state and year, so errors are correlated within a year and
# the std_err / p_value / CI overstate precision
pd.concat([run.coefficients.filter(like="pres_party_signed:", axis=0) for run in econ_ablations[1:]]).round(3)

In [ ]:
# Each relative indicator against the existing specification it refines (that one as the base), on the same rows:
# MAE, RMSE, bias and winner accuracy per subset; the better value of each pair is bold
RELATIVE_VS_EXISTING = {  # relative indicator: existing specification
    "pres_x_log_rel_gas_price": "pres_x_gas_region_minus_us",              # log ratio vs $ gap, gas price level
    "pres_x_gas_region_minus_us_yoy": "pres_x_gas_region_yoy",             # relative vs regional 12-month gas change
    "pres_x_unemp_change_12m_minus_us": "pres_x_unemp_change_12m",         # relative vs state 12-month change
}
econ_by_name = {run.config["name"]: run for run in econ_ablations}
for relative, existing in RELATIVE_VS_EXISTING.items():
    display(tracking.highlight(tracking.compare([econ_by_name[existing], econ_by_name[relative]],
                                                subsets=COMPARE_SUBSETS)))

## Summary: classic_ols and the best variants

For each section, on its own subset and pooled over the eval years: `classic_ols`, then the `SUMMARY_TOP` best variants of each ablation block by `SUMMARY_RANK_BY` (ranked within that subset, so the picks can differ by section). Each block also shows its base when that base's rows differ from the main `classic_ols`: the FEC block leaves out the districts FEC has no data for, and the small-dollar block trains from 2008.

- **Metrics:** `n_train` (training rows), `n` (scored districts in the subset), MAE, RMSE, bias (mean residual, actual − predicted: + means Democrats outperformed the forecast), accuracy, precision D / R, missed races and missed %.
- **Δ columns:** variant − **its own block's base**, which scores exactly the same rows. ✓ (green) marks an improvement and ✗ (red) a regression: lower MAE, RMSE, |bias| and misses, higher accuracy and precision.
- **Bias:** signed bias is shown as is, but its Δ is `Δ|bias|` = |variant bias| − |base bias|, so a negative value always means less bias, whichever party the base under-predicted.
- **Coefficients:** one row per run shown, blank where a run doesn't use the term.

In [ ]:
from IPython.display import Markdown

SUMMARY_BLOCKS = {  # each block's base first, as above
    "ablations": ablations,
    "fec": fec_ablations,
    "small_dollar": small_dollar_ablations,
    "econ": econ_ablations,
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
SUMMARY_RANK_BY = "mae"  # lowest first (also rmse, bias by |bias|, missed_pct; accuracy_pct / precision_*: highest)
SUMMARY_TOP = 2

for section, subset in SUMMARY_SECTIONS.items():
    summary, coefficients = tracking.best_of_blocks(SUMMARY_BLOCKS, subset=subset, rank_by=SUMMARY_RANK_BY,
                                                    top=SUMMARY_TOP)
    display(Markdown(f"### {section} (`{subset}`): top {SUMMARY_TOP} per block by {SUMMARY_RANK_BY}"))
    display(tracking.style_best_of(summary))  # ✓ improvement / ✗ regression vs the block's base
    display(coefficients.style.format(precision=3, na_rep="-"))

## Rolling backtest: classic_ols and the best variants

The summary above fits once, on `TRAIN_YEARS`, and scores 2018–2022 together. Here every ablation block is refit in each **expanding window** (`ROLLING_WINDOWS`) and scored on that window's single eval year:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

The small-dollar block trains from 2008 in every window, since `small_dollar_share_diff` has no 2006 data. Each block shares its rows across its runs in every window (`require_complete`, as above).

**Averages give each window equal weight.** `bias` is the mean signed bias (+ means Democrats outperformed the forecast). `|bias|` is the mean of each window's |bias|, so opposite misses in different years don't cancel, and `Δ|bias|` uses it. `mae_2018`, `mae_2020` and `mae_2022` show each window. The ranking, Δ columns and ✓ / ✗ marks work as in the summary above, on the averages. Coefficients are averaged over the windows.

In [ ]:
import warnings

ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
ROLLING_BLOCKS = {  # block: (formulas, base first; shared columns; first training year)
    "ablations": (ABLATIONS, ABLATION_SHARED_COLUMNS, 2006),
    "fec": (FEC_ABLATIONS, FEC_SHARED_COLUMNS, 2006),
    "small_dollar": (SMALL_DOLLAR_ABLATIONS, SMALL_DOLLAR_SHARED_COLUMNS, 2008),  # no small-dollar data in 2006
    "econ": (ECON_ABLATIONS, ECON_SHARED_COLUMNS, 2006),
}

# the missing-predictor warnings repeat the blocks' warnings above for every window; each run's exclusions are in
# its .excluded and .counts
with warnings.catch_warnings(action="ignore"):
    rolling = {
        eval_year: {
            block: [
                run_ols(
                    formula=formula,
                    train_years=[year for year in train_years if year >= first_train_year],
                    eval_years=[eval_year],
                    predict_years=[],
                    predictors=predictors,
                    targets=targets,
                    use_holdout=USE_HOLDOUT,
                    require_complete=shared_columns,
                    race_types=EVAL_RACE_TYPES,
                    competitive=COMPETITIVE,
                    name=name,
                    save=False,
                )
                for name, formula in formulas.items()
            ]
            for block, (formulas, shared_columns, first_train_year) in ROLLING_BLOCKS.items()
        }
        for eval_year, train_years in ROLLING_WINDOWS.items()
    }

In [ ]:
for section, subset in SUMMARY_SECTIONS.items():
    summary, coefficients = tracking.rolling_best_of(rolling, subset=subset, rank_by=SUMMARY_RANK_BY,
                                                     top=SUMMARY_TOP)
    display(Markdown(f"### Rolling {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}, "
                     f"top {SUMMARY_TOP} per block by {SUMMARY_RANK_BY}"))
    display(tracking.style_best_of(summary))  # ✓ improvement / ✗ regression vs the block's base
    display(coefficients.style.format(precision=3, na_rep="-"))  # mean over the windows

## Saved runs

Save the runs worth keeping: `tracking.save_run(result)` or `run_experiment(..., save=True)`. Each gets a folder in `results/house/runs/` and a row in `results/house/runs.csv`; the CLI saves by default.

In [ ]:
# run_id = tracking.save_run(result)  # or run_ols(..., save=True)

In [ ]:
runs = tracking.load_runs()  # every saved run; empty until one is saved
if runs.empty:
    print("No saved runs yet: results/house/runs.csv is written by tracking.save_run(result) or run_ols(..., save=True)")
else:
    display(runs.sort_values("mae_valid")[["run_id", "formula", "n_train", "mae_valid", "rmse_valid", "bias_valid",
                                           "missed_races_valid", "mae_competitive_ex_ante", "mae_competitive_ex_post"]])

In [ ]:
if not runs.empty:
    saved = tracking.load_run(runs["run_id"].iloc[-1])  # config, metrics, counts, fit, coefficients, predictions, excluded
    display(saved.coefficients)

In [ ]:
if not runs.empty:
    display(tracking.highlight(tracking.compare(runs["run_id"].tail(3).tolist())))